Accumulate results

In [17]:
import pandas as pd

base_path = "stackPredAMR_Results"

antibiotics = ["Amikacin_AB", "Gentamicin_AB", "Piperacillin-tazobactam_AB"]

combinations = [
    ("mix", "na"),
    ("mix", "phoenix"),
    ("mix", "vitek"),
    ("phoenix", "na"),
    ("phoenix", "phoenix"),
    ("phoenix", "vitek"),
]

results = []

for training, test in combinations:

    for fold in range(5):

        folder = f"Evaluation_CV_Conclusio_" f"{training}Train_{test}Test_fold{fold}"

        file_path = f"{base_path}/{folder}/Evaluation.csv"

        df = pd.read_csv(file_path, index_col=0)

        for antibiotic in antibiotics:
            results.append(
                {
                    "Training": training,
                    "Test": test,
                    "Fold": fold,
                    "Antibiotic": antibiotic,
                    "Accuracy": df.loc[antibiotic, "Accuracy"],
                }
            )

results_df = pd.DataFrame(results)
results_df.to_csv("statistics/Accuracy_combined.csv", index=False)
print(results_df)

   Training   Test  Fold                  Antibiotic  Accuracy
0       mix     na     0                 Amikacin_AB  0.857143
1       mix     na     0               Gentamicin_AB  0.891429
2       mix     na     0  Piperacillin-tazobactam_AB  0.717391
3       mix     na     1                 Amikacin_AB  0.807692
4       mix     na     1               Gentamicin_AB  0.834356
..      ...    ...   ...                         ...       ...
85  phoenix  vitek     3               Gentamicin_AB  0.880000
86  phoenix  vitek     3  Piperacillin-tazobactam_AB  0.784232
87  phoenix  vitek     4                 Amikacin_AB  0.786667
88  phoenix  vitek     4               Gentamicin_AB  0.870000
89  phoenix  vitek     4  Piperacillin-tazobactam_AB  0.808000

[90 rows x 5 columns]


Two sided (mixed vs phoenix) paired mutation test for accuracy

In [18]:
import numpy as np
import pandas as pd
from itertools import product


def analyze_mix_vs_phoenix(
    df_input, n_bootstrap=10000, alternative="two-sided", random_state=42
):

    data = df_input.copy()

    # Checks
    required_columns = {"Training", "Test", "Fold", "Antibiotic", "Accuracy"}
    missing = required_columns - set(data.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    if alternative not in {"two-sided", "greater"}:
        raise ValueError("alternative must be 'two-sided' or 'greater'")

    data = data[data["Training"].isin(["mix", "phoenix"])].copy()
    duplicates = data.groupby(["Training", "Test", "Fold", "Antibiotic"]).size()
    if (duplicates > 1).any():
        raise ValueError(
            "There are duplicate observations for at least one "
            "Training × Test × Fold × Antibiotic combination."
        )

    pivot = data.pivot_table(
        index=["Test", "Fold", "Antibiotic"],
        columns="Training",
        values="Accuracy",
        aggfunc="first",
    )
    if "mix" not in pivot.columns or "phoenix" not in pivot.columns:
        raise ValueError("Both 'mix' and 'phoenix' must be present.")
    incomplete = pivot[pivot[["mix", "phoenix"]].isna().any(axis=1)]
    if not incomplete.empty:
        raise ValueError(
            "Some Antibiotic × Test × Fold combinations do not "
            "contain both mix and phoenix results:\n"
            f"{incomplete}"
        )

    pivot["Difference"] = pivot["mix"] - pivot["phoenix"]

    # Aggregate the antibiotics within each Test and Fold block
    block_df = (
        pivot.reset_index()
        .groupby(["Test", "Fold"], as_index=False)
        .agg(
            mix_accuracy=("mix", "median"),
            phoenix_accuracy=("phoenix", "median"),
            difference=("Difference", "median"),
            n_antibiotics=("Antibiotic", "nunique"),
        )
    )

    # Meta data
    n_antibiotics = int(block_df["n_antibiotics"].iloc[0])
    observed_difference = block_df["difference"].mean()
    differences = block_df["difference"].to_numpy()
    n_blocks = len(differences)

    # Permutations
    n_permutations = 2**n_blocks
    permutation_statistics = np.empty(n_permutations, dtype=float)
    # Exact enumeration of all sign combinations
    for i, signs in enumerate(product([-1, 1], repeat=n_blocks)):
        signs = np.asarray(signs)
        permutation_statistics[i] = np.mean(differences * signs)

    if alternative == "greater":
        # H1: MIX > PHOENIX
        p_value = np.sum(permutation_statistics >= observed_difference) / n_permutations
    else:
        # H1: MIX != PHOENIX
        p_value = (
            np.sum(np.abs(permutation_statistics) >= abs(observed_difference))
            / n_permutations
        )

    # Confidence intervall
    rng = np.random.default_rng(random_state)

    bootstrap_differences = np.empty(n_bootstrap, dtype=float)

    for i in range(n_bootstrap):

        sample = rng.choice(differences, size=n_blocks, replace=True)

        bootstrap_differences[i] = np.mean(sample)

    ci_lower, ci_upper = np.percentile(bootstrap_differences, [2.5, 97.5])

    result = {
        "n_blocks": n_blocks,
        "n_test_datasets": block_df["Test"].nunique(),
        "n_folds": block_df["Fold"].nunique(),
        "n_antibiotics": n_antibiotics,
        "mean_mix_accuracy": (block_df["mix_accuracy"].mean()),
        "mean_phoenix_accuracy": (block_df["phoenix_accuracy"].mean()),
        "mean_difference": observed_difference,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "p_value": p_value,
        "alternative": alternative,
        "significant_0.05": p_value < 0.05,
        "n_permutations": n_permutations,
    }

    return result


accuracy_combined = pd.read_csv("statistics/Accuracy_combined.csv")
result_output = analyze_mix_vs_phoenix(
    accuracy_combined, n_bootstrap=10000, alternative="two-sided", random_state=42
)
print(result_output)


result_output_df = pd.DataFrame(
    list(result_output.items()), columns=["Metric", "Result"]
)
result_output_df.to_csv("statistics/results.csv", index=False)

{'n_blocks': 15, 'n_test_datasets': 3, 'n_folds': 5, 'n_antibiotics': 3, 'mean_mix_accuracy': np.float64(0.8707743849418303), 'mean_phoenix_accuracy': np.float64(0.8129908511860333), 'mean_difference': np.float64(0.052561048588120725), 'ci_lower': np.float64(0.030007362657685218), 'ci_upper': np.float64(0.07567291591522723), 'p_value': np.float64(0.00067138671875), 'alternative': 'two-sided', 'significant_0.05': np.True_, 'n_permutations': 32768}
